<a href="https://colab.research.google.com/github/DhariniVenki/Self-Improving-SLM/blob/main/notebooks/run_a_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run A: baseline evaluation

Setup tasks 2, 8 and 9: load Qwen2.5-Coder-0.5B-Instruct, check it writes code,
then score it on the MBPP test (500) and heldout (60) splits.

Use a GPU runtime (Runtime → Change runtime type → T4 GPU). Results are saved
to Google Drive, so if Colab disconnects, rerun the cells: finished parts are skipped.

In [5]:
!nvidia-smi

Fri Oct 30 15:48:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [6]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/self-improving-slm'   # shared results folder

Mounted at /content/drive


In [7]:
import os
%cd /content
if os.path.exists('Self-Improving-SLM'):
    !cd Self-Improving-SLM && git pull
else:
    !git clone https://github.com/DhariniVenki/Self-Improving-SLM.git
%cd /content/Self-Improving-SLM

/content
Cloning into 'Self-Improving-SLM'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 34 (delta 2), reused 25 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 22.26 KiB | 22.26 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Self-Improving-SLM


vLLM is roughly 10x faster than Hugging Face generation, which matters for pass@8
(4,000 samples). If installing it causes errors, set `USE_VLLM = False`; everything
still works, just slower.

In [13]:
USE_VLLM = True

!pip -q install -r requirements.txt
if USE_VLLM:
    !pip -q install vllm
BACKEND = 'vllm' if USE_VLLM else 'hf'

## Unit tests for the sandbox and code extraction

In [9]:
!python -m unittest discover tests

...................
----------------------------------------------------------------------
Ran 19 tests in 6.538s

OK


## Task 2: does the model load and write code?

In [10]:
!python scripts/check_model.py --backend {BACKEND}

config.json: 100% 659/659 [00:00<00:00, 2.61MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 22.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 38.5MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 119MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 148MB/s]
(MainProcess pid=2682) INFO 10-30 15:53:20 [api_utils.py:286] non-default args: {'dtype': 'bfloat16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Coder-0.5B-Instruct'}
(MainProcess pid=2682) INFO 10-30 15:53:36 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=2682) INFO 10-30 15:53:36 [model.py:2152] Using max model len 2048
(MainProcess pid=2682) INFO 10-30 15:53:36 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=2682) INFO 10-30 15:53:36 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_an

## Task 9: Run A

Greedy pass@1 on test and heldout, plus 8 samples per problem for pass@8 (finding 5).

In [11]:
!python scripts/evaluate.py --run A --splits test heldout --pass-at-k 8 --backend {BACKEND} --output-dir {OUTPUT_DIR}

(MainProcess pid=4180) INFO 10-30 15:56:46 [api_utils.py:286] non-default args: {'dtype': 'bfloat16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Coder-0.5B-Instruct'}
(MainProcess pid=4180) INFO 10-30 15:56:46 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=4180) INFO 10-30 15:56:46 [model.py:2152] Using max model len 2048
(MainProcess pid=4180) INFO 10-30 15:56:46 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=4180) INFO 10-30 15:56:46 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files: 100% 9/9 [00:00<00:00, 1863.77it/s]
Download complete:           |  0.00B            
Download complete:           |  0.00B            
Recons

## Results

In [12]:
import json, glob
for path in sorted(glob.glob(f'{OUTPUT_DIR}/A/*/*/summary.json')):
    s = json.load(open(path))
    extra = {k: round(v, 4) for k, v in s.items() if k.startswith('pass@') and k != 'pass@1'}
    print(f"{s['split']:8s} {s['mode']:8s} pass@1={s['pass@1']:.4f} {extra} "
          f"solved>=1: {s['problems_solved_at_least_once']}/{s['n_problems']}  "
          f"shown-pass-hidden-fail: {s['shown_pass_hidden_fail']}  ({s['seconds']}s)")
    print('          ', s['status_counts'])

heldout  greedy   pass@1=0.4500 {} solved>=1: 27/60  shown-pass-hidden-fail: 8  (1.2s)
           {'pass': 27, 'no_code': 0, 'syntax': 0, 'runtime': 4, 'wrong_answer': 29, 'timeout': 0}
heldout  sample8  pass@1=0.3937 {'pass@8': 0.6167} solved>=1: 37/60  shown-pass-hidden-fail: 49  (7.6s)
           {'pass': 189, 'no_code': 0, 'syntax': 3, 'runtime': 54, 'wrong_answer': 234, 'timeout': 0}
test     greedy   pass@1=0.3300 {} solved>=1: 165/500  shown-pass-hidden-fail: 39  (11.7s)
           {'pass': 165, 'no_code': 0, 'syntax': 1, 'runtime': 54, 'wrong_answer': 276, 'timeout': 4}
test     sample8  pass@1=0.3080 {'pass@8': 0.538} solved>=1: 269/500  shown-pass-hidden-fail: 286  (79.8s)
           {'pass': 1232, 'no_code': 0, 'syntax': 38, 'runtime': 522, 'wrong_answer': 2194, 'timeout': 14}
